# **Validation of pyR0compute with published models**

This notebook reproduces the validation table of the pyR0compute paper. Each model is written as it appears in its source, $\mathcal{R}_0$ is computed with `R0Model` and compared with the published expression.

**Sources**

* van den Driessche, P., Watmough, J. (2002). Reproduction numbers and sub-threshold endemic equilibria for compartmental models of disease transmission. *Math. Biosci.* 180, 29-48. https://doi.org/10.1016/S0025-5564(02)00108-6
* Ciupe, S.M., Heffernan, J.M. (2017). In-host modeling. *Infect. Dis. Model.* 2, 188-202. https://doi.org/10.1016/j.idm.2017.04.002
* Cuesta-Herrera, L., Pastenes, L., Arencibia, A.D., Córdova-Lepe, F., Montoya, C. (2024). Dynamics of activation and regulation of the immune response to attack by viral pathogens using mathematical modeling. *Mathematics* 12, 2681. https://doi.org/10.3390/math12172681
* Cuesta-Herrera, L., Pastenes, L., Córdova-Lepe, F., Arencibia, A.D. (2025). Mathematical modeling of the immune response mediated by human T-helper lymphocytes in viral diseases. *Math. Biosci. Eng.* 22(11), 2807-2825. https://doi.org/10.3934/mbe.2025103

**Matching criterion**

| Case | Criterion |
|---|---|
| General | the symbolic difference between the computed and the published $\mathcal{R}_0$ simplifies to 0 |
| Results with a square root | the squares are compared |
| Multistrain model | the set of nonzero eigenvalues of $FV^{-1}$ is compared |
| Equilibrium with a square root (Ciupe and Heffernan, Eq. 3) | compact form and numerical value at 36 parameter sets |
| Seven-compartment model (Cuesta-Herrera et al., 2024) | compact form, substituting the equilibrium values used in the source |

Times are the median of three runs of `R0Model` and `R0_compact`. The final table is saved in `outputs/07_validation.csv`.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Yvan-BM-Git/pyR0compute/blob/main/examples/07_validation.ipynb)

In [1]:
%pip install pyR0compute                      # Install pyR0compute from PyPI

In [2]:
import os                                     # Folder for the table
import time                                   # Timing
import platform                               # Machine information
import statistics                             # Median of the times
import warnings                               # Warnings of the library
import numpy as np                            # Numerical computation
import pandas as pd                           # Results table
import sympy as sp                            # Symbolic computation
import pyr0compute                            # Library version
from pyr0compute import R0Model               # Main class of the library
from pyr0compute import parse_expression as P # Text -> SymPy expression

os.makedirs("outputs", exist_ok=True)         # The table is saved in outputs/
warnings.simplefilter("ignore")               # The warnings are discussed in the text
print("pyR0compute", pyr0compute.__version__, "| SymPy", sp.__version__,
      "| Python", platform.python_version(), "|", platform.processor() or platform.machine())

results = []                                  # One row per validation case


def same(a, b):
    """True if a - b simplifies to 0."""
    return sp.simplify(sp.sympify(a) - sp.sympify(b)) == 0


def validate(case, source, text, infected, published, compare=None, **options):
    """Compute R0, compare it with the published one and record time and assumptions."""
    times = []
    for _ in range(3):                        # Median of three runs
        t0 = time.perf_counter()
        model = R0Model(text, infected=infected, **options)
        model.R0_compact
        times.append(time.perf_counter() - t0)
    match = compare(model) if compare else same(model.R0_compact, published)
    assumptions = model.check_assumptions()   # (A1)-(A5) of van den Driessche and Watmough
    results.append({
        "case": case, "source": source,
        "n": len(model.variables), "m": len(infected),
        "match": bool(match),
        "t_s": round(statistics.median(times), 2),
        **{a: assumptions[a].status for a in ("A1", "A2", "A3", "A4", "A5")},
    })
    print(f"{case}: match = {match}, t = {statistics.median(times):.2f} s")
    display(model.R0_compact)
    return model

pyR0compute 0.3.1 | SymPy 1.14.0 | Python 3.13.16 | x86_64


## 1. Epidemiological models of van den Driessche and Watmough (2002)

### 1.1 Treatment model (Section 4.1)

Infected compartments $E$ and $I$. Treatment failure $pr_2I$ is a transition, not a new infection. The source obtains
$$\mathcal{R}_0=\frac{\beta_1\nu}{(d+\nu+r_1)(d+r_2)-\nu p r_2}.$$
Its Table 1 shows two alternative decompositions: (a) treatment failure as a new infection and (b) also progression $\nu E$. They are declared with `new_infections`.

In [3]:
TREATMENT = """
    N = S + E + I + T
    dE/dt = beta1*S*I/N + beta2*T*I/N - (d + nu + r1)*E + p*r2*I
    dI/dt = nu*E - (d + r2)*I
    dS/dt = b - d*S - beta1*S*I/N
    dT/dt = -d*T + r1*E + q*r2*I - beta2*T*I/N
"""
validate("Treatment", "vdW 2002, 4.1", TREATMENT, ["E", "I"],
         P("beta1*nu/((d + nu + r1)*(d + r2) - nu*p*r2)"))

validate("Treatment, Table 1(a)", "vdW 2002, 4.1", TREATMENT, ["E", "I"],
         P("nu*(beta1 + p*r2)/((d + nu + r1)*(d + r2))"),
         new_infections={"E": "beta1*S*I/N + beta2*T*I/N + p*r2*I", "I": "0"})

validate("Treatment, Table 1(b)", "vdW 2002, 4.1", TREATMENT, ["E", "I"],
         None,                                # Result with a square root: compare squares
         compare=lambda m: same(m.R0**2, P("nu*(beta1 + p*r2)/((d + nu + r1)*(d + r2))")),
         new_infections={"E": "beta1*S*I/N + beta2*T*I/N + p*r2*I", "I": "nu*E"});

Treatment: match = True, t = 0.18 s


-beta1*nu/(-d**2 - d*nu - d*r1 - d*r2 + nu*p*r2 - nu*r2 - r1*r2)

Treatment, Table 1(a): match = True, t = 0.19 s


nu*(beta1 + p*r2)/((d + r2)*(d + nu + r1))

Treatment, Table 1(b): match = True, t = 0.18 s


sqrt(nu*(beta1 + p*r2))/sqrt((d + r2)*(d + nu + r1))

### 1.2 Staged progression (Section 4.3), with four stages

The last stage does not transmit. The source obtains (Eq. 13)
$$\mathcal{R}_0=\frac{\beta_1}{\nu_1+d_1}+\frac{\beta_2\nu_1}{(\nu_1+d_1)(\nu_2+d_2)}+\frac{\beta_3\nu_1\nu_2}{(\nu_1+d_1)(\nu_2+d_2)(\nu_3+d_3)}.$$

In [4]:
validate("Staged progression", "vdW 2002, 4.3", """
    N = S + I1 + I2 + I3 + I4
    dI1/dt = (beta1*I1 + beta2*I2 + beta3*I3)*S/N - (v1 + d1)*I1
    dI2/dt = v1*I1 - (v2 + d2)*I2
    dI3/dt = v2*I2 - (v3 + d3)*I3
    dI4/dt = v3*I3 - d4*I4
    dS/dt = b - b*S - (beta1*I1 + beta2*I2 + beta3*I3)*S/N
""", ["I1", "I2", "I3", "I4"],
    P("beta1/(v1 + d1) + beta2*v1/((v1 + d1)*(v2 + d2)) + beta3*v1*v2/((v1 + d1)*(v2 + d2)*(v3 + d3))"));

Staged progression: match = True, t = 0.30 s


(beta1*d2*d3 + beta1*d2*v3 + beta1*d3*v2 + beta1*v2*v3 + beta2*d3*v1 + beta2*v1*v3 + beta3*v1*v2)/((d1 + v1)*(d2 + v2)*(d3 + v3))

### 1.3 Multistrain model with superinfection (Section 4.4)

$FV^{-1}$ has two eigenvalues, $\beta_i/(b+\gamma_i)$, and $\mathcal{R}_0$ is the larger one (Eqs. 16 and 17). The term $\nu I_1 I_2$ is a transfer between infected compartments.

In [5]:
b, g1, g2, b1, b2 = sp.symbols("b gamma1 gamma2 beta1 beta2")
validate("Multistrain", "vdW 2002, 4.4", """
    dI1/dt = beta1*I1*S - (b + gamma1)*I1 + nu*I1*I2
    dI2/dt = beta2*I2*S - (b + gamma2)*I2 - nu*I1*I2
    dS/dt = b - b*S + gamma1*I1 + gamma2*I2 - (beta1*I1 + beta2*I2)*S
""", ["I1", "I2"], None,
    compare=lambda m: {sp.simplify(e) for e in m.eigenvalues} == {b1/(b + g1), b2/(b + g2)});

Multistrain: match = True, t = 0.13 s


Max(beta1/(b + gamma1), beta2/(b + gamma2))

### 1.4 Vector-host model (Section 4.5)

The source obtains (Eq. 21) $\mathcal{R}_0=\sqrt{\beta_s\beta_m/(c(b+\gamma))}$; the squares are compared.

In [6]:
validate("Vector-host", "vdW 2002, 4.5", """
    dI/dt = beta_s*S*V - (b + gamma)*I
    dV/dt = beta_m*M*I - c*V
    dS/dt = b - b*S + gamma*I - beta_s*S*V
    dM/dt = c - c*M - beta_m*M*I
""", ["I", "V"], None,
    compare=lambda m: same(m.R0**2, P("beta_s*beta_m/(c*(b + gamma))")));

Vector-host: match = True, t = 0.09 s


sqrt(beta_m*beta_s)/sqrt(c*(b + gamma))

## 2. Within-host models of Ciupe and Heffernan (2017)

### 2.1 Chronic infection, Eq. (1)

$\mathcal{R}_0 = p\beta T_0/(\delta c)$ with $T_0 = s/d$ (Eq. 2).

In [7]:
validate("Chronic infection", "CH 2017, Eq. (1)", """
    dT/dt = s - d*T - beta*T*V
    dI/dt = beta*T*V - delta*I
    dV/dt = p*I - c*V
""", ["I", "V"], P("p*beta*(s/d)/(delta*c)"));

Chronic infection: match = True, t = 0.05 s


beta*p*s/(c*d*delta)

### 2.2 Hepatocyte proliferation and cure, Eq. (3)

The equilibrium $T_0$ is the positive root of a quadratic, and the source obtains (Eq. 4)
$$\mathcal{R}_0=\frac{p\beta T_0}{c\left[\rho+\delta+r_I\left(T_0/T_m-1\right)\right]}.$$
The library returns $\mathcal{R}_0$ in terms of $T^{*}$; the compact form is compared with $T^{*}=T_0$, and so is the numerical value at 36 parameter sets with a positive denominator.

In [8]:
def compare_eq3(m):
    T_star = next(s for s in m.R0_compact.free_symbols if s.name == "T_star")
    s_, rT, rI, Tm, d, beta, p, c, delta, rho = sp.symbols("s r_T r_I T_m d beta p c delta rho")
    compact = m.R0_compact.xreplace({x: sp.Symbol(x.name) for x in m.R0_compact.free_symbols})
    form = same(compact, p*beta*sp.Symbol("T_star")/(c*(rho + delta + rI*(sp.Symbol("T_star")/Tm - 1))))
    rng = np.random.default_rng(1)
    ok, n = True, 0
    while n < 36:
        v = {k: float(10**rng.uniform(-2, 1)) for k in "s r_T r_I T_m d beta p c delta rho".split()}
        T0 = v["T_m"]/(2*v["r_T"])*(v["r_T"] - v["d"] + np.sqrt((v["r_T"] - v["d"])**2 + 4*v["s"]*v["r_T"]/v["T_m"]))
        den = v["rho"] + v["delta"] + v["r_I"]*(T0/v["T_m"] - 1)
        if den <= 0:
            continue                          # Outside the range where V is an M-matrix
        published = v["p"]*v["beta"]*T0/(v["c"]*den)
        ok &= abs(m.R0_numeric(v) - published) <= 1e-9*published
        n += 1
    print("compact form:", form, "| 36 numerical points:", ok)
    return form and ok

eq3 = validate("Hepatocyte proliferation", "CH 2017, Eq. (3)", """
    dT/dt = s + r_T*T*(1 - (T + I)/T_m) - d*T - beta*T*V + rho*I
    dI/dt = beta*T*V + r_I*I*(1 - (T + I)/T_m) - delta*I - rho*I
    dV/dt = p*I - c*V
""", ["I", "V"], None, compare=compare_eq3)
eq3.dfe_definitions                           # T* in closed form

compact form: True | 36 numerical points: True


Hepatocyte proliferation: match = True, t = 0.21 s


T_star*T_m*beta*p/(c*(T_star*r_I + T_m*delta - T_m*r_I + T_m*rho))

{T_star: (sqrt(T_m)*sqrt(T_m*d**2 - 2*T_m*d*r_T + T_m*r_T**2 + 4*r_T*s) - T_m*(d - r_T))/(2*r_T)}

### 2.3 Acute infection with a latent phase, Eq. (6)

Without production or death of target cells, every state $(N,0,0,0)$ is a disease-free equilibrium: the equilibrium is declared with `dfe={"T": "N"}`. The source states that $\mathcal{R}_0$ is that of Eq. (2) with $T_0=N$.

In [9]:
validate("Acute infection with latency", "CH 2017, Eq. (6)", """
    dT/dt = -beta*T*V
    dE/dt = beta*T*V - k*E
    dI/dt = k*E - delta*I
    dV/dt = p*I - c*V
""", ["E", "I", "V"], P("p*beta*N/(delta*c)"), dfe={"T": "N"});

Acute infection with latency: match = True, t = 0.05 s


N*beta*p/(c*delta)

### 2.4 Self-replicating pathogen, Eq. (7) with $f(V)=rV(1-V/K)$

The source gives two numbers (Eqs. 12 and 13): $\mathcal{R}_{0,I}=\dfrac{p}{\delta}\dfrac{\beta T_0}{c-r}$, if the infection starts with an infected cell, and $\mathcal{R}_{0,V}=\dfrac{r}{c}+\dfrac{p\beta T_0}{c\delta}$, if it starts with a pathogen particle. The automatic rule keeps the logistic growth in $\mathcal{V}$ and produces $\mathcal{R}_{0,I}$; declared as a new infection, it produces $\mathcal{R}_{0,V}$.

In [10]:
SELF_REPLICATION = """
    dT/dt = s - d*T - beta*T*V
    dI/dt = beta*T*V - delta*I
    dV/dt = r*V*(1 - V/K) + p*I - c*V
"""
validate("Self-replication, R0,I", "CH 2017, Eq. (7)", SELF_REPLICATION, ["I", "V"],
         P("p/delta*beta*(s/d)/(c - r)"))
validate("Self-replication, R0,V", "CH 2017, Eq. (7)", SELF_REPLICATION, ["I", "V"],
         P("r/c + p*beta*(s/d)/(c*delta)"),
         new_infections={"I": "beta*T*V", "V": "r*V*(1 - V/K)"});

Self-replication, R0,I: match = True, t = 0.14 s


-beta*p*s/(d*delta*(-c + r))

Self-replication, R0,V: match = True, t = 0.06 s


(beta*p*s + d*delta*r)/(c*d*delta)

### 2.5 Pathogen loss through infection, Eq. (14)

$\mathcal{R}_{0,loss}=\dfrac{\beta T_0}{c+\beta T_0}\dfrac{p}{\delta}$ (Eq. 16).

In [11]:
validate("Pathogen loss", "CH 2017, Eq. (14)", """
    dT/dt = s - d*T - beta*T*V
    dI/dt = beta*T*V - delta*I
    dV/dt = p*I - c*V - beta*T*V
""", ["I", "V"], P("beta*(s/d)*p/((c + beta*s/d)*delta)"));

Pathogen loss: match = True, t = 0.12 s


beta*p*s/(delta*(beta*s + c*d))

## 3. Immune response models of Cuesta-Herrera et al.

### 3.1 Virus without T lymphocytes and with helper T lymphocytes (2025, systems 2.1 and 2.2)

$\mathcal{R}_0=\kappa(\lambda/d)(\nu/\mu)/\alpha$ and $\mathcal{R}_0^{*}=\dfrac{\kappa\lambda\nu/d}{(\alpha+\beta b/c)(\mu+\tau b/c)}$.

In [12]:
validate("Virus without T lymphocytes", "CH-2025, (2.1)", """
    dE/dt = lambda - d*E - kappa*E*V
    dI/dt = kappa*E*V - alpha*I
    dV/dt = nu*I - mu*V
""", ["I", "V"], P("kappa*(lambda/d)*(nu/mu)/alpha"))

validate("Helper T lymphocytes", "CH-2025, (2.2)", """
    dE/dt = lambda - d*E - kappa*E*V
    dI/dt = kappa*E*V - alpha*I - beta*Th*I
    dV/dt = nu*I - mu*V - tau*Th*V
    dTh/dt = b - c*Th + gamma*I*Th
""", ["I", "V"], P("(kappa*lambda*nu/d)/((alpha + beta*b/c)*(mu + tau*b/c))"));

Virus without T lymphocytes: match = True, t = 0.05 s


kappa*lambda*nu/(alpha*d*mu)

Helper T lymphocytes: match = True, t = 0.16 s


c**2*kappa*lambda*nu/(d*(alpha*c + b*beta)*(b*tau + c*mu))

### 3.2 Regulatory T lymphocytes (2024, system 3)

Seven compartments. The equilibrium is solved in four blocks, $[E]\to[H,R]\to[C]\to[B]$, and $\mathcal{R}_0$ is written in terms of $B^{*}$ and $C^{*}$. Eq. (4) of the source is recovered by substituting $C^{*}=g_C/d_C$ and $B^{*}=g_B/d_B$, which are the equilibrium values only when $\tau_C=\rho_C=\tau_B=\rho_B=0$; with these parameters positive, $C^{*}$ and $B^{*}$ depend on $H^{*}$ and $R^{*}$.

In [13]:
PUBLISHED_2024 = P("g_E*(g_B + d_B*z_BV)*(g_C + d_C*z_CI)*nu*tau_E"
                   "/(d_E*(d_I*(g_C + d_C*z_CI) + g_C*tau_I)*(d_V*(g_B + d_B*z_BV) + g_B*tau_V)*z_VE)")

def compare_2024(m):
    R = m.R0_compact
    star = {s.name: s for s in R.free_symbols}
    substituted = R.subs({star["C_star"]: P("g_C/d_C"), star["B_star"]: P("g_B/d_B")})
    names = {s.name: s for s in PUBLISHED_2024.free_symbols}
    substituted = substituted.xreplace({s: names.get(s.name, s) for s in substituted.free_symbols})
    return same(substituted, PUBLISHED_2024)

ch24 = validate("Regulatory T lymphocytes", "CH-2024, (3)", """
    dE/dt = g_E - (d_E + tau_E*V/(V + z_VE))*E
    dI/dt = tau_E*V*E/(V + z_VE) - (d_I + tau_I*C/(C + z_CI))*I
    dV/dt = nu*I - (d_V + tau_V*B/(B + z_BV))*V
    dH/dt = g_H - (d_H - tau_H*I/(I + z_IH) + rho_H*R/(R + z_RH))*H
    dC/dt = g_C - (d_C - tau_C*H/(H + z_HC) + rho_C*R/(R + z_RC))*C
    dB/dt = g_B - (d_B - tau_B*H/(H + z_HB) + rho_B*R/(R + z_RB))*B
    dR/dt = g_R - (d_R - tau_R*H/(H + z_HR) + rho_R)*R
""", ["I", "V"], None, compare=compare_2024)
for block in ch24.dfe_blocks:                 # Equilibrium blocks and time of each
    print(block["variables"], block["status"], f'{block["seconds"]:.2f} s')

Regulatory T lymphocytes: match = True, t = 6.99 s


g_E*nu*tau_E*(B_star + z_BV)*(C_star + z_CI)/(d_E*z_VE*(B_star*d_V + B_star*tau_V + d_V*z_BV)*(C_star*d_I + C_star*tau_I + d_I*z_CI))

['E'] closed form 0.00 s
['H', 'R'] closed form 5.83 s
['C'] closed form 0.01 s
['B'] closed form 0.01 s


## 4. Validation table

`match` states whether the computed $\mathcal{R}_0$ matches the published one under the criterion of each case. Columns A1 to A5 are the result of `check_assumptions()`: `holds`, `fails` or `undecided` (could not be decided without parameter values).

In [14]:
table = pd.DataFrame(results)
table.to_csv("outputs/07_validation.csv", index=False)
print(f"{table.match.sum()} of {len(table)} cases match the source")
table

15 of 15 cases match the source


,case,source,n,m,match,t_s,A1,A2,A3,A4,A5
0,Treatment,"vdW 2002, 4.1",4,2,True,0.18,holds,holds,holds,holds,undecided
1,"Treatment, Table 1(a)","vdW 2002, 4.1",4,2,True,0.19,holds,holds,holds,holds,holds
2,"Treatment, Table 1(b)","vdW 2002, 4.1",4,2,True,0.18,holds,holds,holds,holds,holds
3,Staged progression,"vdW 2002, 4.3",5,4,True,0.30,holds,holds,holds,holds,holds
4,Multistrain,"vdW 2002, 4.4",3,2,True,0.13,holds,holds,holds,holds,holds
5,Vector-host,"vdW 2002, 4.5",4,2,True,0.09,holds,holds,holds,holds,holds
6,Chronic infection,"CH 2017, Eq. (1)",3,2,True,0.05,holds,holds,holds,holds,holds
7,Hepatocyte proliferation,"CH 2017, Eq. (3)",3,2,True,0.21,holds,holds,holds,holds,undecided
8,Acute infection with latency,"CH 2017, Eq. (6)",4,3,True,0.05,holds,holds,holds,holds,fails
9,"Self-replication, R0,I","CH 2017, Eq. (7)",3,2,True,0.14,holds,holds,holds,holds,undecided
